In [0]:



-- Display arrondissements table 

SELECT * 
FROM workspace.geojson_files.arrondissements ;

-- Display chantiers table 

SELECT * 
FROM workspace.geojson_files.chantiers_perturbants ; 

-- Display fermetures voiries table 

SELECt * 
FROM workspace.geojson_files.fermetures_voirie ; 

-- Display quartiers table 

SELECT * 
FROM workspace.geojson_files.quartier_paris ; 

-- Display velib 

SELECT * 
FROM workspace.geojson_files.velib_emplacement_des_stations ;

-- neighborhood count 

SELECT  COUNT(*) AS neighborhood_count 
FROM workspace.geojson_files.quartier_paris ; 

-- Vélib’ station records in the dataset

SELECT COUNT(*) AS stations_count
FROM workspace.geojson_files.velib_emplacement_des_stations ;

-- geometry type for each neighborhood

SELECT l_qu , ST_GeometryType (geom) AS geometry_type  
FROM workspace.geojson_files.quartier_paris ; 

--  perimeter of each neighborhood polygon. 

SELECT l_qu , ST_Perimeter(ST_Transform(geom,2154)) AS perimeter 
FROM workspace.geojson_files.quartier_paris 
ORDER BY perimeter DESC ; 

--  area of each neighborhood polygon 

SELECT l_qu,ST_Area(geom) AS area 
FROM workspace.geojson_files.quartier_paris 
ORDER BY area DESC ; 

--  area of each neighborhood polygon in m2

SELECT l_qu,ST_Area(ST_Transform(geom,2154)) AS area 
FROM workspace.geojson_files.quartier_paris 
ORDER BY area DESC ; 

-- comparaison between the provided surface with the geometry-derived area

SELECT l_qu,
    ROUND(surface,2),
    ROUND(ST_Area(ST_Transform(geom,2154)),2),
    CASE WHEN  ROUND(surface,2) == (ROUND(ST_Area(ST_Transform(geom,2154)),2)) THEN TRUE
    WHEN ROUND(surface,2) != (ROUND(ST_Area(ST_Transform(geom,2154)),2)) THEN FALSE END AS equal 
FROM workspace.geojson_files.quartier_paris ;

-- centroid of each neighborhood. 

SELECT l_qu , ST_AsText(ST_Centroid(geom)) AS centroid 
FROM workspace.geojson_files.quartier_paris ; 

-- Vélib’ stations count inside each neighborhood polygon 

SELECT q.l_qu , COUNT (stationcode) AS stations_count
FROM workspace.geojson_files.velib_emplacement_des_stations s  
JOIN workspace.geojson_files.quartier_paris q on st_within(s.geom ,q.geom) 
GROUP BY q.l_qu 
ORDER BY stations_count DESC ; 

--   List the station names and neighborhood names for stations inside a neighborhood.

SELECT q.l_qu,s.name AS station_name , s.stationcode 
FROM workspace.geojson_files.quartier_paris q 
LEFT JOIN workspace.geojson_files.velib_emplacement_des_stations s ON ST_Contains(q.geom,s.geom) 
ORDER BY q.l_qu;

-- station points that intersect a particular neighborhood

SELECT * 
FROM workspace.geojson_files.velib_emplacement_des_stations s
JOIN workspace.geojson_files.quartier_paris q ON ST_Intersects(s.geom,q.geom) 
WHERE q.l_qu = 'Odéon';

-- stations count in each arrondissement 

SELECT c_ar,COUNT ( stationcode )  AS stations_count
FROM workspace.geojson_files.arrondissements a 
JOIN workspace.geojson_files.velib_emplacement_des_stations s ON ST_Contains(a.geom,s.geom)
GROUP BY c_ar 
ORDER BY stations_count DESC;

-- Chantiers geometry type 

SELECT identifiant, ST_GeometryType(geom)
FROM workspace.geojson_files.chantiers_perturbants ; 

-- worksite in neighberhoods 

SELECT l_qu,COUNT ( identifiant ) AS chantiers_count
FROM workspace.geojson_files.chantiers_perturbants c 
JOIN workspace.geojson_files.quartier_paris q ON ST_within(c.geom,q.geom)
GROUP BY l_qu 
ORDER BY chantiers_count DESC;